# Mapa completo: tudo o que dá para obter

**O que este notebook faz:** percorre **todas as bases** que a biblioteca PySUS
oferece e mostra, com dados atuais do catálogo, o que existe em cada uma —
período coberto, estados e grupos de informação.

Use-o como referência: quando quiser saber "existe dado de X?", rode este
notebook em vez de procurar na documentação.

As nove bases da PySUS não são tudo: a **atenção primária** fica fora delas,
e a última seção mostra por onde se chega a ela.

**Tempo estimado:** 4 a 6 minutos (só consulta catálogos, quase não baixa dados).

## Preparação

In [2]:
%pip install pysus==2.10.6 nest_asyncio -q
import nest_asyncio
nest_asyncio.apply()
print("Ambiente pronto.")

Note: you may need to restart the kernel to use updated packages.
Ambiente pronto.


## As nove bases da biblioteca

| Função | Base | Recorte exigido |
|---|---|---|
| `sinan()` | Doenças de notificação | agravo + ano (nacional) |
| `sim()` | Mortalidade | estado + ano |
| `sinasc()` | Nascidos vivos | estado + ano |
| `sih()` | Internações hospitalares | estado + ano + mês |
| `sia()` | Produção ambulatorial | estado + ano + mês |
| `cnes()` | Estabelecimentos, leitos, profissionais | estado + ano + mês + **grupo** |
| `pni()` | Imunizações | estado + ano |
| `ciha()` | Atendimentos além do SUS | estado + ano + mês + `group=None` |
| `ibge()` | População e indicadores sociais | ano |

## Levantamento automático

A célula abaixo consulta o catálogo de cada base e resume o que há.

In [3]:
from pysus import list_files
import pandas as pd

BASES = ["sinan", "sim", "sinasc", "sih", "sia", "cnes", "pni", "ciha", "ibge"]

resumo = []
for base in BASES:
    catalogo = list_files(dataset=base)
    nomes = [n.split("\\")[-1] for n in catalogo["name"]]
    anos = sorted({int(a) for a in catalogo["year"].dropna()}) if "year" in catalogo else []
    ufs = {u for u in catalogo["state"].dropna()} if "state" in catalogo else set()
    largura = 4 if base in ("sinan", "ibge") else 2
    grupos = sorted({n[:largura] for n in nomes})

    resumo.append({
        "Base": base.upper(),
        "Arquivos": len(catalogo),
        "De": anos[0] if anos else "-",
        "Até": anos[-1] if anos else "-",
        "UFs": len(ufs),
        "Grupos": len(grupos),
    })

pd.DataFrame(resumo)

,Base,Arquivos,De,Até,UFs,Grupos
0,SINAN,1149,1999,2026,1,61
1,SIM,1315,1979,2026,29,2
2,SINASC,871,1994,2026,29,2
3,SIH,31512,1992,2026,28,7
4,SIA,57465,1994,2026,29,13
5,CNES,80512,2005,2026,27,14
6,PNI,1523,1994,2026,30,3
7,CIHA,4722,2011,2026,26,1
8,IBGE,152,1980,2070,0,8


## Os grupos de cada base

Cada base divide seus dados em **grupos** — conjuntos com colunas diferentes.
Saber qual grupo você precisa evita baixar centenas de megabytes à toa.

In [4]:
GRUPOS_CONHECIDOS = {
    "cnes": {
        "LT": "Leitos", "ST": "Estabelecimentos", "PF": "Profissionais",
        "EQ": "Equipamentos", "SR": "Serviços especializados",
        "HB": "Habilitações", "EP": "Equipes", "IN": "Incentivos",
        "RC": "Regras contratuais", "DC": "Dados complementares",
        "GM": "Gestão e metas", "EF": "Estabelecimentos filantrópicos",
        "EE": "Estabelecimento de ensino",
    },
    "sih": {
        "RD": "AIH reduzida (uma linha por internação)",
        "SP": "Serviços profissionais", "RJ": "AIH rejeitada",
        "ER": "AIH com erro", "CH": "Cadastro hospitalar",
        "CM": "Comunicação de movimentação",
    },
    "sia": {
        "PA": "Produção ambulatorial (principal)",
        "BI": "Boletim de produção individualizada",
        "AB": "APAC de cirurgia bariátrica", "AM": "APAC de medicamentos",
        "AN": "APAC de nefrologia", "AQ": "APAC de quimioterapia",
        "AR": "APAC de radioterapia", "AD": "APAC de laudos diversos",
        "AT": "APAC de tratamento", "AC": "APAC de confecção de fístula",
        "PS": "RAAS psicossocial", "SA": "Serviços especializados",
    },
    "sim": {"DO": "Declarações de óbito"},
    "sinasc": {"DN": "Declarações de nascido vivo", "DNR": "Retroativos"},
    "pni": {"CP": "Cobertura por município", "DP": "Doses aplicadas"},
    "ciha": {"CIHA": "Comunicação de internação hospitalar e ambulatorial"},
    "ibge": {
        "POPT": "População total", "POPB": "População por faixas",
        "PROJ": "Projeções populacionais", "ESCA": "Escolaridade",
        "ESCB": "Escolaridade (detalhe)", "ALFB": "Alfabetização",
        "REND": "Renda", "IDOS": "População idosa",
    },
}

for base, grupos in GRUPOS_CONHECIDOS.items():
    print(f"\n{base.upper()}")
    for codigo, descricao in grupos.items():
        print(f"   {codigo:<5} {descricao}")

CNES
   LT    Leitos
   ST    Estabelecimentos
   PF    Profissionais
   EQ    Equipamentos
   SR    Serviços especializados
   HB    Habilitações
   EP    Equipes
   IN    Incentivos
   RC    Regras contratuais
   DC    Dados complementares
   GM    Gestão e metas
   EF    Estabelecimentos filantrópicos
   EE    Estabelecimento de ensino
SIH
   RD    AIH reduzida (uma linha por internação)
   SP    Serviços profissionais
   RJ    AIH rejeitada
   ER    AIH com erro
   CH    Cadastro hospitalar
   CM    Comunicação de movimentação
SIA
   PA    Produção ambulatorial (principal)
   BI    Boletim de produção individualizada
   AB    APAC de cirurgia bariátrica
   AM    APAC de medicamentos
   AN    APAC de nefrologia
   AQ    APAC de quimioterapia
   AR    APAC de radioterapia
   AD    APAC de laudos diversos
   AT    APAC de tratamento
   AC    APAC de confecção de fístula
   PS    RAAS psicossocial
   SA    Serviços especializados
SIM
   DO    Declarações de óbito
SINASC
   DN    Declar

## Os agravos do SINAN

O SINAN é a única base organizada por doença. Estes são os agravos com dados
no ano mais recente:

In [5]:
ANO = 2024
catalogo = list_files(dataset="sinan", year=ANO)
agravos = sorted({n.split("\\")[-1][:4] for n in catalogo["name"]})

NOMES = {
    "DENG": "Dengue", "CHIK": "Chikungunya", "ZIKA": "Zika",
    "TUBE": "Tuberculose", "HANS": "Hanseníase", "LEPT": "Leptospirose",
    "MALA": "Malária", "MENI": "Meningite", "VIOL": "Violência",
    "ANIM": "Acidente por animal peçonhento", "ACGR": "Acidente de trabalho grave",
    "ACBI": "Acidente biológico", "ESQU": "Esquistossomose",
    "CHAG": "Doença de Chagas", "LEIV": "Leishmaniose visceral",
    "LTAN": "Leishmaniose tegumentar", "HEPA": "Hepatites virais",
    "SIFA": "Sífilis adquirida", "SIFC": "Sífilis congênita",
    "SIFG": "Sífilis em gestante", "COQU": "Coqueluche",
    "DIFT": "Difteria", "TETA": "Tétano acidental", "RAIV": "Raiva",
    "FMAC": "Febre maculosa", "FTIF": "Febre tifoide", "BOTU": "Botulismo",
    "COLE": "Cólera", "HANT": "Hantavirose", "INFL": "Influenza",
    "PEST": "Peste", "TOXC": "Toxoplasmose congênita",
    "TRAC": "Tracoma", "VARC": "Varicela", "PNEU": "Pneumoconiose",
    "PAIR": "Perda auditiva", "LERD": "LER/DORT", "DERM": "Dermatoses",
    "IEXO": "Intoxicação exógena", "MENT": "Transtorno mental",
    "NTRA": "Notificação de trabalho", "CANC": "Câncer relacionado ao trabalho",
    "SDTA": "Surto de doença transmitida por alimentos",
}

print(f"{len(agravos)} agravos com dados em {ANO}:\n")
for codigo in agravos:
    print(f"   {codigo:<6} {NOMES.get(codigo, '(consulte o Ministério da Saúde)')}")

56 agravos com dados em 2024:
   ACBI   Acidente biológico
   ACGR   Acidente de trabalho grave
   AIDA   (consulte o Ministério da Saúde)
   AIDC   (consulte o Ministério da Saúde)
   ANIM   Acidente por animal peçonhento
   ANTR   (consulte o Ministério da Saúde)
   BOTU   Botulismo
   CANC   Câncer relacionado ao trabalho
   CHAG   Doença de Chagas
   CHIK   Chikungunya
   COLE   Cólera
   COQU   Coqueluche
   DCRJ   (consulte o Ministério da Saúde)
   DENG   Dengue
   DERM   Dermatoses
   DIFT   Difteria
   ESQU   Esquistossomose
   EXAN   (consulte o Ministério da Saúde)
   FMAC   Febre maculosa
   FTIF   Febre tifoide
   HANS   Hanseníase
   HANT   Hantavirose
   HIVA   (consulte o Ministério da Saúde)
   HIVC   (consulte o Ministério da Saúde)
   HIVE   (consulte o Ministério da Saúde)
   HIVG   (consulte o Ministério da Saúde)
   IEXO   Intoxicação exógena
   LEIV   Leishmaniose visceral
   LEPT   Leptospirose
   LERD   LER/DORT
   LTAN   Leishmaniose tegumentar
   MALA   Malár

## Consultando qualquer recorte

A função abaixo responde à pergunta "existe dado de X?" para qualquer combinação:

In [6]:
def o_que_existe(base, **filtros):
    """Mostra o que há no catálogo para o recorte pedido."""
    catalogo = list_files(dataset=base, **filtros)
    if len(catalogo) == 0:
        print(f"❌ Nada em {base} com {filtros}")
        return
    nomes = sorted(n.split("\\")[-1] for n in catalogo["name"])
    print(f"✅ {len(nomes)} arquivo(s) em {base} com {filtros}")
    for nome in nomes[:8]:
        print(f"     {nome}")
    if len(nomes) > 8:
        print(f"     … e mais {len(nomes) - 8}")


o_que_existe("cnes", state="PR", year=2024, month=12)
print()
o_que_existe("sim", state="AM", year=2023)

✅ 12 arquivo(s) em cnes com {'state': 'PR', 'year': 2024, 'month': 12}
     DCPR2412.parquet
     EFPR2412.parquet
     EPPR2412.parquet
     EQPR2412.parquet
     GMPR2412.parquet
     HBPR2412.parquet
     INPR2412.parquet
     LTPR2412.parquet
     … e mais 4


✅ 2 arquivo(s) em sim com {'state': 'AM', 'year': 2023}
     DO23OPEN.parquet
     DOAM2023.parquet


## E a atenção primária?

Repare no que **não** apareceu na lista acima: a atenção primária. Nem consulta
de enfermeiro, nem visita de agente comunitário, nem cadastro de família, nem os
indicadores do Previne Brasil.

Não é omissão da PySUS. É que esse registro não passa pelos sistemas que ela
cobre. A produção da APS é feita no **e-SUS APS**, na própria unidade de saúde, e
sai pelo **SISAB** — que não publica arquivo para baixar como o DATASUS faz, e
sim relatórios num site.

A PySUS tem uma função com esse nome, e ela é uma armadilha: existe, roda, não dá
erro e devolve **nada**.

In [7]:
import pysus

vazias = []

# AVISO MEDIDO EM 30/08/2026, e nao teorico: rodar esta celula derruba o seu
# acesso ao sisab.saude.gov.br por mais de uma hora. Depois dela, ate um GET
# simples da pagina inicial do SISAB falha com RemoteProtocolError em 10,2
# segundos certinhos — constante demais para ser congestionamento — enquanto o
# apidadosabertos.saude.gov.br continua respondendo 200 em 0,2s. Reproduzido em
# teste A/B: sozinho, o notebook do SISAB acerta 6 de 6; depois desta celula,
# erra todas.
#
# Tentamos espacar as onze chamadas com time.sleep(2) e NAO adiantou: a rajada
# de verdade acontece DENTRO da pysus, que pagina a API a cada chamada. Nao ha
# conserto barato aqui, e fingir que ha seria pior que o problema. O que fica e
# o aviso: se voce vai usar o notebook do SISAB hoje, rode-o ANTES desta celula.
for nome in ["atencao_primaria", "bnafar", "vacinacao", "assistencia_saude",
             "prevencao_promocao", "ouvidoria", "saude_indigena", "covid19",
             "vigilancia_meio_ambiente", "sisvan", "sisagua"]:
    funcao = getattr(pysus, nome, None)
    if funcao is None:
        continue
    try:
        r = funcao()
        vazias.append((nome, len(r) if hasattr(r, "__len__") else -1))
    except Exception as erro:
        vazias.append((nome, f"erro: {type(erro).__name__}"))

sem_nada = [n for n, q in vazias if q == 0]
print("Funções da origem 'Saúde' (portal de dados abertos):\n")
for nome, quantos in vazias:
    marca = "vazia" if quantos == 0 else f"{quantos} conjunto(s)"
    print(f"   {nome:26} {marca}")
print(f"\n{len(sem_nada)} de {len(vazias)} devolvem lista vazia — sem erro, sem aviso.")
print("Se você chamar atencao_primaria() e não vier nada, o problema não é seu.")
print()
print("AVISO: esta célula acabou de derrubar o seu acesso ao "
      "sisab.saude.gov.br por mais de uma hora (medido em 30/08/2026). Se for "
      "usar o notebook do SISAB hoje, rode-o antes desta célula.")


Funções da origem 'Saúde' (portal de dados abertos):
   atencao_primaria           vazia
   bnafar                     vazia
   vacinacao                  vazia
   assistencia_saude          vazia
   prevencao_promocao         vazia
   ouvidoria                  vazia
   saude_indigena             vazia
   covid19                    vazia
   vigilancia_meio_ambiente   vazia
   sisvan                     2 conjunto(s)
   sisagua                    10 conjunto(s)
9 de 11 devolvem lista vazia — sem erro, sem aviso.
Se você chamar atencao_primaria() e não vier nada, o problema não é seu.
AVISO: esta célula acabou de derrubar o seu acesso ao sisab.saude.gov.br por mais de uma hora (medido em 30/08/2026). Se for usar o notebook do SISAB hoje, rode-o antes desta célula.


## Por onde a atenção primária realmente sai

Há quatro caminhos, todos fora da PySUS, e cada um chega a um nível diferente de
detalhe. Esta tabela é o mapa que faltava:

| O que você quer | Por onde | Granularidade | Exemplo pronto |
|---|---|---|---|
| Equipes e profissionais cadastrados | **CNES**, pela PySUS | estabelecimento → equipe (INE) → profissional | `AtencaoPrimaria/ubs-no-mapa-e-o-que-o-cadastro-esconde` |
| Produção: consultas, visitas, procedimentos | **SISAB**, Relatório de Validação | UBS × **equipe** × mês | `AtencaoPrimaria/producao-da-ubs-no-sisab-equipe-por-equipe` |
| Indicadores do Previne Brasil | **API de dados abertos** do Ministério | município × quadrimestre | `AtencaoPrimaria/previne-brasil-indicadores-da-aps` |
| Programas: Saúde na Escola, Bucal, cadastro | **Dados abertos do DEMAS** | município | `AtencaoPrimaria/painel-da-aps-programas-do-municipio` |

O caminho mais fino é o do SISAB: ele desce até **a equipe**, dentro da unidade,
mês a mês. É o nível em que uma coordenação de UBS de fato trabalha.

E o CNES é a ponte entre os dois mundos: ele dá o **INE** (Identificador
Nacional de Equipe), que é a chave para casar o cadastro com a produção do
SISAB.

### Uma armadilha de sigla, no CNES

`EP` é **Equipes** e `EQ` é **Equipamentos** — o contrário do que a sigla
sugere. Quem pedir `group="EQ"` procurando equipe recebe raio-X e
eletrocardiógrafo, sem erro nenhum.

In [8]:
# O CNES é a única porta da APS que passa pela PySUS. Veja o que ele entrega:
UF_APS = "AC"     # troque pelo seu estado

equipes = pysus.cnes(UF_APS, 2025, 1, group="EP", as_dataframe=True)
print(f"CNES/EP — equipes cadastradas em {UF_APS}, jan/2025")
print(f"   {len(equipes):,} equipes")
print(f"   em {equipes['CNES'].nunique():,} estabelecimentos")
print(f"   de {equipes['CODUFMUN'].nunique()} municípios")
print(f"   em {equipes['TIPO_EQP'].nunique()} tipos de equipe diferentes")

# O INE são os últimos dígitos do IDEQUIPE — é ele que liga ao SISAB.
equipes["INE"] = (equipes["IDEQUIPE"].astype(str).str.strip()
                  .str[-8:].str.zfill(10))
print(f"\n   INE (chave para o SISAB): {equipes['INE'].nunique():,} distintos")
print("   exemplos:", equipes[["NOME_EQP", "INE"]].head(3).to_dict("records"))

CNES/EP — equipes cadastradas em AC, jan/2025
   571 equipes
   em 245 estabelecimentos
   de 22 municípios
   em 13 tipos de equipe diferentes
   INE (chave para o SISAB): 571 distintos
   exemplos: [{'NOME_EQP': 'ESB ZULMIRA GARCIA', 'INE': '0002280795'}, {'NOME_EQP': 'URBANA 2', 'INE': '0002096307'}, {'NOME_EQP': 'URBANA', 'INE': '0000004294'}]


> **Por que isso importa agora.** A Portaria GM/MS 7.639/2025 tirou o SISAB da
> lista de bancos de alimentação obrigatória e pôs o **Siaps** no lugar. Ou
> seja: este mapa vale hoje, e a porta da atenção primária deve mudar de nome
> nos próximos anos. Quando mudar, o notebook do SISAB é o primeiro a avisar —
> ele falha alto em vez de devolver tabela vazia.

## Resumo prático

| Se você quer saber… | Use |
|---|---|
| Casos de uma doença | `sinan()` |
| Óbitos e suas causas | `sim()` |
| Nascimentos, cesáreas, peso | `sinasc()` |
| Internações, custos, permanência | `sih()` |
| Consultas, exames, procedimentos | `sia()` |
| Leitos, hospitais, profissionais | `cnes()` |
| Vacinação e cobertura | `pni()` |
| Atendimentos fora do SUS | `ciha()` |
| População (para calcular taxas) | `ibge()` |
| **Atenção primária: produção por equipe** | **SISAB** (fora da PySUS) |
| **Atenção primária: equipes cadastradas** | **`cnes(group="EP")`** |
| **Atenção primária: indicadores do Previne** | **API de dados abertos** |


## Verificação de sanidade

Este notebook é um mapa, e mapa errado é pior que mapa nenhum. As conferências
abaixo falham alto se o catálogo sair do ar, se uma base ficar vazia ou se a
PySUS mudar de comportamento entre uma versão e outra.


In [9]:

print("Verificações\n")
falhas = []

quadro = pd.DataFrame(resumo).set_index("Base")

# 1. nenhuma base pode vir vazia: catalogo fora do ar aparece assim
# "vazias" ja existe, da secao da APS: outro nome, para nao sobrescrever.
bases_vazias = quadro[quadro["Arquivos"] == 0].index.tolist()
print(f"1. Todas as {len(quadro)} bases têm arquivo? "
      f"{'confere' if not bases_vazias else 'ATENÇÃO: vazias — ' + ', '.join(bases_vazias)}")
falhas += [] if not bases_vazias else ["base vazia"]

# 2. as maiores continuam sendo as maiores
maiores = quadro["Arquivos"].nlargest(2).index.tolist()
esperado = {"CNES", "SIA"}
print(f"2. As duas maiores bases são CNES e SIA? {maiores} — "
      f"{'confere' if set(maiores) == esperado else 'ATENÇÃO: mudou muito'}")
falhas += [] if set(maiores) == esperado else ["tamanhos"]

# 3. o catalogo chega ao ano corrente
import datetime
ano_agora = datetime.date.today().year
ate = quadro.loc[quadro.index != "IBGE", "Até"]
recentes = sum(1 for a in ate if isinstance(a, int) and a >= ano_agora - 1)
print(f"3. Bases com dado de {ano_agora - 1} ou depois: {recentes} de {len(ate)} — "
      f"{'confere' if recentes >= len(ate) - 1 else 'ATENÇÃO: catálogo parado'}")
falhas += [] if recentes >= len(ate) - 1 else ["catalogo parado"]

# 4. o portal de dados abertos responde? Esta conferencia ja foi decorativa:
# dizia "confere (e o estado conhecido)" sem condicao NENHUMA, e disse isso
# tanto com 1 vazia (30/08/2026) quanto com 9 (31/08 e 01/09) — o numero
# quadruplicou e o teste nao piscou. O numero de funcoes vazias VARIA com a
# saude do portal do Ministerio; vazio nao e erro da pysus. Mas TODAS vazias
# significa portal fora do ar, e isso o teste precisa gritar.
com_resposta = len(vazias) - len(sem_nada)
portal_vivo = com_resposta > 0
print(f"4. Funções da origem 'Saúde' com resposta: {com_resposta} "
      f"de {len(vazias)} — "
      f"{'confere (o número varia com a saúde do portal)' if portal_vivo
         else 'ATENÇÃO: portal de dados abertos fora do ar'}")
falhas += [] if portal_vivo else ["portal fora do ar"]

# 5. o CNES ainda e a ponte para a APS, com o INE
tem_ine = "INE" in equipes.columns and equipes["INE"].nunique() > 0
print(f"5. CNES/EP entrega equipes com INE: "
      f"{equipes['INE'].nunique():,} identificadores — "
      f"{'confere' if tem_ine else 'ATENÇÃO: a ponte com o SISAB quebrou'}")
falhas += [] if tem_ine else ["ine"]

# 6. EP e equipe, EQ e equipamento — se inverterem, tudo aqui muda
print("6. Lembrete conferido no código da PySUS: EP = Equipes, EQ = Equipamentos")

print()
if falhas:
    print("ATENÇÃO:", falhas, "— o mapa pode estar desatualizado, investigue.")
else:
    print("Tudo confere. Este mapa reflete o catálogo de hoje.")


Verificações
1. Todas as 9 bases têm arquivo? confere
2. As duas maiores bases são CNES e SIA? ['CNES', 'SIA'] — confere
3. Bases com dado de 2025 ou depois: 8 de 8 — confere
4. Funções da origem 'Saúde' com resposta: 2 de 11 — confere (o número varia com a saúde do portal)
5. CNES/EP entrega equipes com INE: 571 identificadores — confere
6. Lembrete conferido no código da PySUS: EP = Equipes, EQ = Equipamentos
Tudo confere. Este mapa reflete o catálogo de hoje.


---
*Notebook do projeto [PySusNoCode](https://github.com/cartaproale/PySusNoCode) —
um produto [Kraemer Academy](https://kraemeracademy.net).
Validado com dados reais do DATASUS.*